# Sentiment Classification using Naive Bayes

## BOW vs TF-IDF

This notebook performs text preprocessing, Bag-of-Words and TF-IDF representation, Multinomial Naive Bayes classification, prediction on test and unseen reviews, and accuracy comparison.

In [ ]:
import pandas as pd
import re
import nltk

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score

## 1. Load the Dataset

In [ ]:
# If reviews.csv is in the same folder as this notebook
df = pd.read_csv("reviews.csv")

print("Dataset:")
print(df)

## 2. Download NLTK Resources

In [ ]:
nltk.download('punkt')
nltk.download('stopwords')

## 3. Text Preprocessing

Steps: lowercase → remove punctuation/special characters → tokenize → remove stop words.

In [ ]:
stop_words = set(stopwords.words('english'))

def preprocess(text):
    # Convert to lowercase
    text = text.lower()

    # Remove punctuation and special characters
    text = re.sub(r'[^a-zA-Z\s]', '', text)

    # Tokenize
    tokens = word_tokenize(text)

    # Remove stop words
    tokens = [word for word in tokens if word not in stop_words]

    # Convert tokens back to text
    return ' '.join(tokens)

df['cleaned_review'] = df['review'].apply(preprocess)

print("Preprocessed Reviews:")
print(df[['review', 'cleaned_review']])

## 4. Split the Dataset into Training and Testing Data

In [ ]:
X = df['cleaned_review']
y = df['sentiment']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

print('Training samples:', len(X_train))
print('Testing samples:', len(X_test))

## 5. Bag-of-Words Representation

In [ ]:
bow_vectorizer = CountVectorizer()

X_train_bow = bow_vectorizer.fit_transform(X_train)
X_test_bow = bow_vectorizer.transform(X_test)

print('BOW training matrix shape:', X_train_bow.shape)
print('BOW testing matrix shape:', X_test_bow.shape)

## 6. Display the BOW Vocabulary

In [ ]:
vocabulary = bow_vectorizer.get_feature_names_out()

print('BOW Vocabulary:')
print(vocabulary)

## 7. Train Multinomial Naive Bayes using BOW

In [ ]:
bow_model = MultinomialNB()
bow_model.fit(X_train_bow, y_train)

print('BOW Naive Bayes model trained successfully.')

## 8. Display Class Prior Probabilities

In [ ]:
print('Class Prior Probabilities:')

for class_name, log_probability in zip(
    bow_model.classes_,
    bow_model.class_log_prior_
):
    probability = 2.718281828 ** log_probability
    print(class_name, ':', probability)

## 9. Display Feature Log Probabilities

In [ ]:
feature_log_prob = pd.DataFrame(
    bow_model.feature_log_prob_,
    columns=bow_vectorizer.get_feature_names_out(),
    index=bow_model.classes_
)

print('Feature Log Probabilities:')
print(feature_log_prob)

## 10. Predict the Classes of Test Reviews

In [ ]:
bow_predictions = bow_model.predict(X_test_bow)

print('BOW Test Predictions:')

for review, actual, predicted in zip(X_test, y_test, bow_predictions):
    print('\nReview:', review)
    print('Actual:', actual)
    print('Predicted:', predicted)

## 11. BOW + Naive Bayes Accuracy

In [ ]:
bow_accuracy = accuracy_score(y_test, bow_predictions)

print('BOW + Naive Bayes Accuracy:', bow_accuracy)
print('BOW + Naive Bayes Accuracy (%):', bow_accuracy * 100)

## 12. Test 3 New / Unseen Reviews

In [ ]:
new_reviews = [
    'The movie was amazing and wonderful',
    'The film was boring and terrible',
    'I enjoyed the brilliant acting'
]

new_reviews_cleaned = [preprocess(review) for review in new_reviews]
new_reviews_bow = bow_vectorizer.transform(new_reviews_cleaned)
new_predictions_bow = bow_model.predict(new_reviews_bow)

print('New / Unseen Reviews - BOW:')

for review, prediction in zip(new_reviews, new_predictions_bow):
    print('Review:', review)
    print('Predicted:', prediction)
    print()

## 13. TF-IDF Representation

In [ ]:
tfidf_vectorizer = TfidfVectorizer()

X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

print('TF-IDF training matrix shape:', X_train_tfidf.shape)
print('TF-IDF testing matrix shape:', X_test_tfidf.shape)

## 14. Train Multinomial Naive Bayes using TF-IDF

In [ ]:
tfidf_model = MultinomialNB()
tfidf_model.fit(X_train_tfidf, y_train)

print('TF-IDF Naive Bayes model trained successfully.')

## 15. Predict Test Reviews using TF-IDF

In [ ]:
tfidf_predictions = tfidf_model.predict(X_test_tfidf)

print('TF-IDF Test Predictions:')

for review, actual, predicted in zip(X_test, y_test, tfidf_predictions):
    print('\nReview:', review)
    print('Actual:', actual)
    print('Predicted:', predicted)

## 16. TF-IDF + Naive Bayes Accuracy

In [ ]:
tfidf_accuracy = accuracy_score(y_test, tfidf_predictions)

print('TF-IDF + Naive Bayes Accuracy:', tfidf_accuracy)
print('TF-IDF + Naive Bayes Accuracy (%):', tfidf_accuracy * 100)

## 17. Test New / Unseen Reviews using TF-IDF

In [ ]:
new_reviews_tfidf = tfidf_vectorizer.transform(new_reviews_cleaned)
new_predictions_tfidf = tfidf_model.predict(new_reviews_tfidf)

print('New / Unseen Reviews - TF-IDF:')

for review, prediction in zip(new_reviews, new_predictions_tfidf):
    print('Review:', review)
    print('Predicted:', prediction)
    print()

## 18. Compare BOW and TF-IDF Predictions

In [ ]:
comparison = pd.DataFrame({
    'Review': new_reviews,
    'BOW Prediction': new_predictions_bow,
    'TF-IDF Prediction': new_predictions_tfidf
})

print('BOW vs TF-IDF Predictions:')
print(comparison)

## 19. Compare Accuracy of Both Approaches

In [ ]:
accuracy_comparison = pd.DataFrame({
    'Method': [
        'BOW + Naive Bayes',
        'TF-IDF + Naive Bayes'
    ],
    'Accuracy': [
        bow_accuracy,
        tfidf_accuracy
    ],
    'Accuracy (%)': [
        bow_accuracy * 100,
        tfidf_accuracy * 100
    ]
})

print('Accuracy Comparison:')
print(accuracy_comparison)

## Conclusion

The notebook compares two text representations—Bag-of-Words and TF-IDF—using Multinomial Naive Bayes. The final accuracy values and predictions are displayed above.